# KV-Cache for Autoregressive Generation

During autoregressive generation, we produce one token at a time. Without KV-cache, we recompute attention over all previous tokens at every step. KV-cache stores past key/value projections to avoid redundant computation.

**Interview questions:**
- "How does KV-cache speed up inference?"
- "What is the memory cost of KV-cache?"
- "When does KV-cache become the bottleneck?"

---

## Key Formulas

**Without KV-cache (step $t$):** recompute all $t$ positions
- Compute: $O(t \cdot d^2)$ for projections + $O(t^2 \cdot d)$ for attention
- Total across $T$ steps: $O(T^2 \cdot d^2 + T^3 \cdot d)$

**With KV-cache (step $t$):** only compute new token
- Compute: $O(d^2)$ for projections + $O(t \cdot d)$ for attention
- Total across $T$ steps: $O(T \cdot d^2 + T^2 \cdot d)$

**Memory per layer:** $2 \cdot B \cdot H \cdot T \cdot d_k$ floats (K and V for each head)

For a model with $L$ layers, $H$ heads, head dim $d_k$, batch $B$, sequence $T$:
$$\text{KV-cache memory} = 2 \cdot L \cdot B \cdot H \cdot T \cdot d_k \cdot \text{bytes\_per\_element}$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Simple Causal Self-Attention (No Cache)

In [ ]:
class CausalSelfAttention(nn.Module):
    """Standard causal self-attention — no caching."""
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        # Causal attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = F.softmax(scores, dim=-1)
        
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(out)

## 2. Causal Self-Attention WITH KV-Cache

Key insight: during generation, at step $t$:
- We only need to compute Q for the **new** token
- K, V for positions $0..t-1$ are unchanged — cache them
- Append new K, V to cache, then attend over the full cache

In [ ]:
class CausalSelfAttentionWithCache(nn.Module):
    """Causal self-attention with KV-cache support."""
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(
        self, 
        x: torch.Tensor,
        kv_cache: tuple[torch.Tensor, torch.Tensor] | None = None,
        use_cache: bool = False,
    ) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor] | None]:
        B, T_new, _ = x.shape
        
        # Project current input
        Q = self.W_q(x).view(B, T_new, self.n_heads, self.d_k).transpose(1, 2)
        K_new = self.W_k(x).view(B, T_new, self.n_heads, self.d_k).transpose(1, 2)
        V_new = self.W_v(x).view(B, T_new, self.n_heads, self.d_k).transpose(1, 2)
        
        # Append to cache if it exists
        if kv_cache is not None:
            K_cached, V_cached = kv_cache
            K = torch.cat([K_cached, K_new], dim=2)  # concat on seq dim
            V = torch.cat([V_cached, V_new], dim=2)
        else:
            K = K_new
            V = V_new
        
        T_total = K.size(2)
        
        # Attention: Q is (B, H, T_new, d_k), K is (B, H, T_total, d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Causal mask: only for the new tokens querying cached positions
        # For generation (T_new=1), no masking needed — the single query can attend to all cached positions
        if T_new > 1:
            causal_mask = torch.triu(
                torch.ones(T_new, T_total, device=x.device), 
                diagonal=T_total - T_new + 1
            ).bool()
            scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, T_new, self.d_model)
        
        new_cache = (K, V) if use_cache else None
        return self.W_o(out), new_cache

## 3. Minimal GPT Block for Testing

In [ ]:
class GPTBlock(nn.Module):
    """Simplified GPT block with optional KV-cache."""
    def __init__(self, d_model: int, n_heads: int, use_kv_cache: bool = False):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        if use_kv_cache:
            self.attn = CausalSelfAttentionWithCache(d_model, n_heads)
        else:
            self.attn = CausalSelfAttention(d_model, n_heads)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        self.use_kv_cache = use_kv_cache
    
    def forward(self, x, kv_cache=None, use_cache=False):
        if self.use_kv_cache:
            attn_out, new_cache = self.attn(self.ln1(x), kv_cache=kv_cache, use_cache=use_cache)
            x = x + attn_out
        else:
            x = x + self.attn(self.ln1(x))
            new_cache = None
        x = x + self.ffn(self.ln2(x))
        return x, new_cache


class MiniGPT(nn.Module):
    """Minimal GPT for benchmarking KV-cache."""
    def __init__(self, vocab_size: int, d_model: int, n_heads: int, 
                 n_layers: int, max_len: int = 512, use_kv_cache: bool = False):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, n_heads, use_kv_cache) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.use_kv_cache = use_kv_cache
    
    def forward(self, idx, kv_caches=None, use_cache=False, pos_offset=0):
        B, T = idx.shape
        positions = torch.arange(pos_offset, pos_offset + T, device=idx.device).unsqueeze(0)
        x = self.token_emb(idx) + self.pos_emb(positions)
        
        new_caches = []
        for i, block in enumerate(self.blocks):
            cache_i = kv_caches[i] if kv_caches is not None else None
            x, new_cache = block(x, kv_cache=cache_i, use_cache=use_cache)
            new_caches.append(new_cache)
        
        logits = self.head(self.ln_f(x))
        return logits, new_caches if use_cache else None

## 4. Generation: Without vs With KV-Cache

In [ ]:
@torch.no_grad()
def generate_no_cache(model, prompt_ids: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
    """Generate tokens by recomputing the full sequence at each step."""
    generated = prompt_ids.clone()
    for _ in range(max_new_tokens):
        # Feed entire sequence through the model
        logits, _ = model(generated)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
    return generated


@torch.no_grad()
def generate_with_cache(model, prompt_ids: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
    """Generate tokens using KV-cache."""
    # Prefill: process entire prompt, build initial cache
    logits, kv_caches = model(prompt_ids, use_cache=True)
    next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
    generated = torch.cat([prompt_ids, next_token], dim=1)
    pos_offset = prompt_ids.size(1)
    
    # Decode: one token at a time, using cache
    for _ in range(max_new_tokens - 1):
        # Only feed the LAST token — cache has everything else
        logits, kv_caches = model(
            next_token, 
            kv_caches=kv_caches, 
            use_cache=True,
            pos_offset=pos_offset
        )
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        pos_offset += 1
    
    return generated

# Quick correctness check: both should produce the same output
torch.manual_seed(42)
model_no_cache = MiniGPT(vocab_size=100, d_model=64, n_heads=4, n_layers=2, use_kv_cache=False)
torch.manual_seed(42)
model_with_cache = MiniGPT(vocab_size=100, d_model=64, n_heads=4, n_layers=2, use_kv_cache=True)

# Copy weights
model_with_cache.load_state_dict(model_no_cache.state_dict(), strict=False)

prompt = torch.randint(0, 100, (1, 5))
out_no = generate_no_cache(model_no_cache, prompt, 10)
out_yes = generate_with_cache(model_with_cache, prompt, 10)

print(f"Without cache: {out_no[0].tolist()}")
print(f"With cache:    {out_yes[0].tolist()}")
print(f"Match: {torch.equal(out_no, out_yes)}")

## 5. Benchmark: Speed Comparison

In [ ]:
def benchmark_generation(vocab_size=500, d_model=128, n_heads=4, n_layers=4, 
                         prompt_len=10, gen_lengths=[10, 20, 50, 100]):
    results = {'no_cache': [], 'with_cache': []}
    
    for gen_len in gen_lengths:
        # No cache model
        torch.manual_seed(0)
        model_nc = MiniGPT(vocab_size, d_model, n_heads, n_layers, 
                           max_len=prompt_len + gen_len + 10, use_kv_cache=False).to(device)
        model_nc.eval()
        prompt = torch.randint(0, vocab_size, (1, prompt_len), device=device)
        
        start = time.perf_counter()
        for _ in range(3):  # average over 3 runs
            _ = generate_no_cache(model_nc, prompt, gen_len)
        t_no = (time.perf_counter() - start) / 3
        results['no_cache'].append(t_no)
        
        # With cache model
        torch.manual_seed(0)
        model_wc = MiniGPT(vocab_size, d_model, n_heads, n_layers,
                           max_len=prompt_len + gen_len + 10, use_kv_cache=True).to(device)
        model_wc.load_state_dict(model_nc.state_dict(), strict=False)
        model_wc.eval()
        
        start = time.perf_counter()
        for _ in range(3):
            _ = generate_with_cache(model_wc, prompt, gen_len)
        t_wc = (time.perf_counter() - start) / 3
        results['with_cache'].append(t_wc)
        
        speedup = t_no / t_wc if t_wc > 0 else float('inf')
        print(f"Generate {gen_len} tokens: no_cache={t_no:.4f}s, with_cache={t_wc:.4f}s, speedup={speedup:.1f}x")
    
    return results, gen_lengths

results, gen_lengths = benchmark_generation()

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(gen_lengths, results['no_cache'], 'ro-', label='No cache')
axes[0].plot(gen_lengths, results['with_cache'], 'bs-', label='With KV-cache')
axes[0].set_xlabel('Generated tokens')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Generation Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

speedups = [nc/wc for nc, wc in zip(results['no_cache'], results['with_cache'])]
axes[1].plot(gen_lengths, speedups, 'g^-', linewidth=2)
axes[1].set_xlabel('Generated tokens')
axes[1].set_ylabel('Speedup')
axes[1].set_title('KV-Cache Speedup Factor')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Speedup increases with sequence length because no-cache cost grows quadratically.")

## 6. Memory Analysis

In [ ]:
def kv_cache_memory_bytes(n_layers, n_heads, head_dim, seq_len, batch_size=1, dtype_bytes=2):
    """Calculate KV-cache memory in bytes.
    
    KV-cache stores K and V tensors for each layer.
    Each: (batch, n_heads, seq_len, head_dim)
    """
    # 2 for K and V, per layer
    return 2 * n_layers * batch_size * n_heads * seq_len * head_dim * dtype_bytes

# Real model examples
models = {
    'LLaMA-7B': {'n_layers': 32, 'n_heads': 32, 'head_dim': 128, 'd_model': 4096},
    'LLaMA-13B': {'n_layers': 40, 'n_heads': 40, 'head_dim': 128, 'd_model': 5120},
    'LLaMA-70B': {'n_layers': 80, 'n_heads': 64, 'head_dim': 128, 'd_model': 8192},
    'GPT-3 175B': {'n_layers': 96, 'n_heads': 96, 'head_dim': 128, 'd_model': 12288},
}

seq_lengths = [1024, 4096, 8192, 32768, 131072]

print(f"{'Model':<15} {'Seq Len':>10} {'KV-Cache (fp16)':>18} {'KV-Cache (int8)':>18}")
print('=' * 65)

for name, cfg in models.items():
    for seq_len in [4096, 32768, 131072]:
        mem_fp16 = kv_cache_memory_bytes(
            cfg['n_layers'], cfg['n_heads'], cfg['head_dim'], seq_len, dtype_bytes=2
        )
        mem_int8 = kv_cache_memory_bytes(
            cfg['n_layers'], cfg['n_heads'], cfg['head_dim'], seq_len, dtype_bytes=1
        )
        print(f"{name:<15} {seq_len:>10,} {mem_fp16/1e9:>15.1f} GB {mem_int8/1e9:>15.1f} GB")
    print()

In [ ]:
# Visualize: KV-cache memory vs model weights
fig, ax = plt.subplots(figsize=(10, 5))

for name, cfg in models.items():
    # Approximate model weight memory (fp16)
    # ~12 * d_model^2 * n_layers params (rough estimate)
    model_params = 12 * cfg['d_model']**2 * cfg['n_layers']
    model_mem_gb = model_params * 2 / 1e9  # fp16
    
    kv_mems = [kv_cache_memory_bytes(cfg['n_layers'], cfg['n_heads'], cfg['head_dim'], sl, dtype_bytes=2) / 1e9 
               for sl in seq_lengths]
    ax.plot(seq_lengths, kv_mems, 'o-', label=f"{name} (weights={model_mem_gb:.0f}GB)")

ax.set_xlabel('Sequence Length')
ax.set_ylabel('KV-Cache Memory (GB, fp16)')
ax.set_title('KV-Cache Memory vs Sequence Length (batch=1)')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("At long sequences, KV-cache can exceed model weight memory!")
print("This is why GQA/MQA (shared KV heads) and KV-cache quantization matter.")

## 7. Pre-allocated KV-Cache (Production Pattern)

Dynamic concatenation (`torch.cat`) is wasteful — it allocates new memory each step. In production, pre-allocate the cache to max length and track the current position.

In [ ]:
class PreallocatedKVCache:
    """Pre-allocated KV-cache for efficient inference.
    
    Allocates full buffer upfront, writes K/V at current position.
    Avoids repeated memory allocation/copy from torch.cat.
    """
    def __init__(self, n_layers: int, batch_size: int, n_heads: int, 
                 max_len: int, head_dim: int, dtype=torch.float16, device='cpu'):
        self.max_len = max_len
        self.current_len = 0
        
        # Pre-allocate: (n_layers, 2, batch, n_heads, max_len, head_dim)
        # 2 for K and V
        self.cache = torch.zeros(
            n_layers, 2, batch_size, n_heads, max_len, head_dim,
            dtype=dtype, device=device
        )
        
        mem_gb = self.cache.nelement() * self.cache.element_size() / 1e9
        print(f"Allocated KV-cache: {self.cache.shape}, {mem_gb:.3f} GB")
    
    def update(self, layer_idx: int, new_k: torch.Tensor, new_v: torch.Tensor) -> tuple:
        """Write new K, V at current position and return the full cache slice."""
        T_new = new_k.size(2)
        start = self.current_len
        end = start + T_new
        assert end <= self.max_len, f"Cache overflow: {end} > {self.max_len}"
        
        self.cache[layer_idx, 0, :, :, start:end, :] = new_k
        self.cache[layer_idx, 1, :, :, start:end, :] = new_v
        
        # Return full valid cache up to current end
        return (
            self.cache[layer_idx, 0, :, :, :end, :],
            self.cache[layer_idx, 1, :, :, :end, :],
        )
    
    def advance(self, n_tokens: int = 1):
        """Advance position counter after processing all layers."""
        self.current_len += n_tokens
    
    def reset(self):
        self.current_len = 0
        self.cache.zero_()

# Demo
cache = PreallocatedKVCache(
    n_layers=4, batch_size=1, n_heads=8, max_len=256, head_dim=64
)

# Simulate prefill (10 tokens)
new_k = torch.randn(1, 8, 10, 64, dtype=torch.float16)
new_v = torch.randn(1, 8, 10, 64, dtype=torch.float16)
k_full, v_full = cache.update(layer_idx=0, new_k=new_k, new_v=new_v)
cache.advance(10)
print(f"After prefill: cache has {cache.current_len} tokens, K shape: {k_full.shape}")

# Simulate decode step (1 token)
new_k = torch.randn(1, 8, 1, 64, dtype=torch.float16)
new_v = torch.randn(1, 8, 1, 64, dtype=torch.float16)
k_full, v_full = cache.update(layer_idx=0, new_k=new_k, new_v=new_v)
cache.advance(1)
print(f"After 1 decode step: cache has {cache.current_len} tokens, K shape: {k_full.shape}")

## 8. Computation Breakdown: Prefill vs Decode

In practice, LLM inference has two distinct phases:

| Phase | Input | Compute | Bottleneck |
|-------|-------|---------|------------|
| **Prefill** | Full prompt (many tokens) | Parallel, compute-bound | GPU FLOPs |
| **Decode** | 1 token at a time | Sequential, memory-bound | Memory bandwidth |

In [ ]:
# Illustrate compute vs memory bottleneck
def estimate_flops_and_bytes(d_model, n_layers, seq_len, batch=1, phase='decode'):
    """Rough estimate of FLOPs and memory bytes for one step."""
    if phase == 'prefill':
        # All tokens processed in parallel
        # QKV projection: 3 * 2 * batch * seq * d^2
        proj_flops = 3 * 2 * batch * seq_len * d_model * d_model * n_layers
        # Attention: 2 * batch * n_heads * seq^2 * d_k
        attn_flops = 2 * batch * seq_len * seq_len * d_model * n_layers
        # FFN: 2 * 2 * batch * seq * d * 4d
        ffn_flops = 2 * 2 * batch * seq_len * d_model * 4 * d_model * n_layers
        total_flops = proj_flops + attn_flops + ffn_flops
        # Weight reads: all weights once
        weight_bytes = (4 * d_model * d_model + 8 * d_model * d_model) * n_layers * 2  # fp16
        return total_flops, weight_bytes
    else:  # decode
        # Single token, but must read full KV-cache
        proj_flops = 3 * 2 * batch * 1 * d_model * d_model * n_layers
        attn_flops = 2 * batch * 1 * seq_len * d_model * n_layers  # attend over cache
        ffn_flops = 2 * 2 * batch * 1 * d_model * 4 * d_model * n_layers
        total_flops = proj_flops + attn_flops + ffn_flops
        # Must read: all weights + KV-cache
        weight_bytes = (4 * d_model * d_model + 8 * d_model * d_model) * n_layers * 2
        kv_bytes = 2 * n_layers * batch * seq_len * d_model * 2  # K+V, fp16
        return total_flops, weight_bytes + kv_bytes

# LLaMA-7B
d, L = 4096, 32
print(f"{'Seq Len':>10} {'Phase':<10} {'TFLOPs':>10} {'Memory (GB)':>12} {'Arith. Intensity':>18}")
print('='*65)
for seq_len in [1024, 4096, 16384]:
    for phase in ['prefill', 'decode']:
        flops, mem_bytes = estimate_flops_and_bytes(d, L, seq_len, phase=phase)
        intensity = flops / mem_bytes  # FLOPs per byte
        print(f"{seq_len:>10} {phase:<10} {flops/1e12:>10.1f} {mem_bytes/1e9:>12.1f} {intensity:>18.1f}")
    print()

print("Low arithmetic intensity = memory-bound (decode is always memory-bound)")
print("High arithmetic intensity = compute-bound (prefill with long sequences)")

## Interview Notes

**"How does KV-cache speed up inference?"**
- Autoregressive generation produces one token at a time
- Without cache: at step $t$, recompute Q, K, V for all $t$ positions = $O(t)$ per step, $O(T^2)$ total
- With cache: store K, V from previous steps, only compute Q/K/V for the new token = $O(1)$ per step for projections
- Still $O(t)$ for attending over the cache, but avoid redundant projection computation
- Net: changes $O(T^2 d^2)$ projection cost to $O(T d^2)$

**"Memory cost of KV-cache?"**
- Per token per layer: $2 \cdot n_{\text{heads}} \cdot d_k \cdot \text{bytes}$ (K and V)
- For LLaMA-7B (fp16): $2 \cdot 32 \cdot 32 \cdot 128 \cdot 2 = 524\text{KB}$ per token
- At 4K context: ~2 GB; at 128K context: ~64 GB (!)
- This is why GQA/MQA exist (reduce K, V heads) and why KV-cache quantization matters

**"When does KV-cache become the bottleneck?"**
- Decode phase is **memory-bandwidth bound**, not compute-bound
- The GPU spends most time reading weights + KV-cache from HBM, not computing
- At long sequences, KV-cache read dominates even weight reads
- Solutions: GQA/MQA, KV-cache quantization (INT8/INT4), PagedAttention (vLLM), sliding window attention